In [ ]:
from datasets import load_dataset, concatenate_datasets

In [ ]:
dataset_names = [
    "weathon/daceflux_nag_10",
    "weathon/daceflux_nag_15",
    "weathon/flux_krea_10",
    "weathon/flux_krea_4",
]

In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
dataset = concatenate_datasets([load_dataset(name)["train"] for name in dataset_names])

README.md:   0%|          | 0.00/397 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/250M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/254M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/397 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/477M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/397 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/497M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset.remove_columns("method")

In [ ]:
methods = ["DanceFlux", "DanceFlux", "Flux Krea", "Flux Krea"]
alpha = [10.0, 15.0, 10.0, 4.0]

In [ ]:
methods = [[i]*300 for i in methods]
methods = [i for j in methods for i in j]
alpha = [[i]*300 for i in alpha]
alpha = [i for j in alpha for i in j]

In [ ]:
len(alpha)

1200

In [ ]:
dataset = dataset.add_column("method", methods)
dataset = dataset.add_column("alpha", alpha)

In [ ]:
import torch
import requests
from PIL import Image
from transformers import BlipProcessor, BlipForImageTextRetrieval

processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-large-coco")
model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-large-coco", torch_dtype=torch.float16).to("cuda")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.78G [00:00<?, ?B/s]

In [ ]:
dataset

Dataset({
    features: ['prompt', 'negative_prompt', 'image', 'method', 'alpha'],
    num_rows: 1200
})

In [ ]:
def blip_rate(batch):
  with torch.no_grad():
    prompt = batch["prompt"]
    inputs = processor(batch["image"], prompt, return_tensors="pt", padding=True).to("cuda", torch.float16)

    itm_scores = model(**inputs)[0]
    itm_scores = itm_scores.softmax(-1)[:,1]
    return {"blip_score": itm_scores.tolist()}

In [ ]:
dataset = dataset.map(blip_rate, batched=True, batch_size=64)

model.safetensors:   0%|          | 0.00/1.78G [00:00<?, ?B/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
np.array(dataset["blip_score"]).mean()

np.float64(0.9170478041966756)

In [ ]:
dataset.save_to_disk("vsf_dataset")

Saving the dataset (0/4 shards):   0%|          | 0/1200 [00:00<?, ? examples/s]

In [ ]:
!pip3 install hpsv3

In [ ]:
from datasets import load_from_disk
dataset = load_from_disk("vsf_dataset")

In [ ]:
from hpsv3 import HPSv3RewardInferencer

inferencer = HPSv3RewardInferencer(device='cuda')

In [ ]:
dataset

Dataset({
    features: ['prompt', 'negative_prompt', 'image', 'method', 'alpha', 'blip_score'],
    num_rows: 1200
})

In [ ]:
def score_hpsv3(sample):
  rewards = inferencer.reward(prompts=[sample["prompt"]], image_paths=[sample["image"]])
  scores = [reward[0].item() for reward in rewards]  # Extract mu values
  return scores[0]

In [ ]:
from datasets import load_dataset
original_ds = load_dataset("weathon/anti_aesthetics_dataset")

README.md:   0%|          | 0.00/442 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
original_ds

DatasetDict({
    train: Dataset({
        features: ['disorted_long_prompt', 'selected', 'desc', 'original_prompt', 'negative_prompt'],
        num_rows: 300
    })
})

In [ ]:
original_ds = original_ds["train"]

In [ ]:
def score_hpsv3_original(sample, i):
  rewards = inferencer.reward(prompts=[original_ds["original_prompt"][i]], image_paths=[sample["image"]])
  scores = [reward[0].item() for reward in rewards]  # Extract mu values
  return scores[0]

In [ ]:
hpsv3_scores = []
for sample in dataset:
  hpsv3_scores.append(score_hpsv3(sample))

In [ ]:
hpsv3_scores_original = []
import tqdm
for i, sample in enumerate(tqdm.tqdm(dataset)):
  hpsv3_scores_original.append(score_hpsv3_original(sample, i % 300))

100%|██████████| 1200/1200 [03:56<00:00,  5.08it/s]


In [ ]:
dataset = dataset.add_column("hpsv3_score_aa", hpsv3_scores)
dataset = dataset.add_column("hpsv3_scores_original", hpsv3_scores_original)


In [ ]:
dataset.push_to_hub("weathon/nag_dataset")

Uploading the dataset shards:   0%|          | 0/4 [00:00<?, ? shards/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   5%|5         | 25.1MB /  497MB            

CommitInfo(commit_url='https://huggingface.co/datasets/weathon/nag_dataset/commit/55b009c8009ef4c94dd5e1dde78b104951b514f9', commit_message='Upload dataset', commit_description='', oid='55b009c8009ef4c94dd5e1dde78b104951b514f9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/weathon/nag_dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='weathon/nag_dataset'), pr_revision=None, pr_num=None)

In [ ]:
import numpy as np
np.where(np.array(hpsv3_scores)<-8)

(array([   6,   37,   47,   64,   83,  306,  347,  364,  383,  526,  551,
         552,  555,  564,  584,  606,  609,  613,  620,  643,  645,  648,
         652,  658,  659,  662,  671,  689,  696,  697,  702,  704,  708,
         713,  719,  720,  728,  738,  753,  760,  761,  768,  770,  778,
         801,  813,  815,  816,  820,  824,  826,  851,  852,  855,  856,
         864,  876,  877,  884,  894, 1068, 1156]),)

In [ ]:
dataset[856]["image"].resize((512, 512))

In [ ]:
idx = [16, 259, 266, 279, 49, 54, 63, 70, 80, 109]